# Diagnose-Fragen: Was sagen die Daten über das *Warum*?
**Internet of Production – KI, Woche 1 (Musterlösung)** · gehört zur Folie „Unsere Fragen: Was kann der Datensatz beantworten?“

In der Gruppenaktivität wurden die Fragen des Vorfertigungsleiters nach *beschreiben – diagnostizieren – vorhersagen – empfehlen – handeln* sortiert und danach in *mit dem Datensatz beantwortbar* und *nicht beantwortbar*. Dieses Notebook beantwortet die sechs **grünen Fragen der Spalte „Diagnostizieren“**:

| # | Frage | Daten |
|---|---|---|
| 1 | Welche Teile verursachen welche Defekte? | `Printproduction_QM_scrap.csv` |
| 2 | Welcher Druckfehler gehört zu welchem Drucker (z. B. Warping)? | QM-Datei |
| 3 | Gibt es einen Zusammenhang zwischen Extruder-Blockade und Verbindungsabbruch? | `Printer_*_error.csv` |
| 4 | Wie viele Teile sind gescheitert, und warum (Defektart)? | Aufträge, QM-Datei, Fehlerdateien |
| 5 | Welche Unterschiede gibt es zwischen den Druckern? | Minutendaten, Fehlerdateien, Aufträge, QM-Datei |
| 6 | Welches Teil auf welchen Drucker, um Ausfälle zu minimieren? | Aufträge, QM-Datei |

**Regel dieses Notebooks:** Jede Antwort nennt die Zahl, ihre **Unsicherheit** und das, was die Daten **nicht** sagen. Diagnose fragt nach Ursachen; ein Beobachtungsdatensatz zeigt Zusammenhänge. Wo die Daten eine Aussage nicht tragen, sagen wir das.

## Vorbereitung und Datenprüfung

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

DATA = Path('../../00_Data/printfarm')
DEF = ['material breakout', 'support structure remains', 'under-extrusion', 'warpage']
rng = np.random.default_rng(0)

orders = pd.read_csv(DATA / 'Printproduction.csv', sep=';', decimal=',').dropna(how='all')
orders['Start'] = pd.to_datetime(orders['Start'], format='%d.%m.%Y %H:%M')
orders['Printer'] = orders['Printer'].astype(int)
orders['failed'] = (orders['Status'] == 'failed').astype(int)
qm = pd.read_csv(DATA / 'Printproduction_QM_scrap.csv', sep=';')

errors = []
for k in (0, 1, 2):
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    e['Printer'] = k
    errors.append(e)
errors = pd.concat(errors)
part_order = sorted(orders['BOM Item'].unique(), key=lambda s: int(s[:-1]))   # 1a, 2a, ..., 10a

print('orders:', len(orders), ' failed:', orders['failed'].sum(), ' planned parts:', int(orders['Lot size'].sum()))
print('orders with a QM record:', qm['Order'].isin(orders['Order']).sum(), ' of', (orders['failed'] == 0).sum(), 'completed orders')
print('QM record for a failed order:', qm['Order'].isin(orders.loc[orders['failed'] == 1, 'Order']).sum())
print('lot size = OK + not OK:', bool((qm['Lot size'] == qm['OK'] + qm['not OK']).all()))
print('not OK = sum of the four defect columns:', bool((qm['not OK'] == qm[DEF].sum(axis=1)).all()))

**Was bedeuten diese Prüfungen für die Analyse?**

- Qualitätskontrolle gibt es nur für **abgeschlossene** Aufträge: 1.001 von 1.028 abgeschlossenen Aufträgen haben einen QM-Eintrag, **kein gescheiterter Auftrag hat einen**. Defekte können gescheiterte Aufträge also nicht erklären; die beiden Verlustarten haben verschiedene Ursachen und müssen getrennt gezählt werden.
- 27 abgeschlossene Aufträge (120 Teile) haben **keinen QM-Eintrag**: Ihre Qualität ist unbekannt, nicht „in Ordnung“.
- Losgröße = OK + nicht OK, die QM-Spalten zählen also **Teile**, nicht Aufträge. Nicht OK = Summe der vier Defektspalten, jedes Ausschussteil hat also **genau eine** Defektart; die Defektarten sind sich ausschließende Kategorien.

## Zwei Arten von Verlust, und warum Unsicherheit Sorgfalt braucht
- **Gescheiterter Auftrag:** Der Auftrag wurde abgebrochen, alle Teile des Loses sind verloren; es gibt **keinen QM-Eintrag**. Der Grund ist eine Störung des Druckers (siehe Frage 4).
- **Ausschussteil:** Der Auftrag wurde abgeschlossen, aber ein Teil ist in der Qualitätskontrolle *nicht in Ordnung*; jedes solche Teil hat genau **einen** Defekt.

Teile eines Auftrags sind **nicht unabhängig** (derselbe Druckauftrag, ein Los aus 3, 6 oder 12 Teilen). Ein klassischer Test, der jedes Teil als unabhängige Beobachtung zählt, ist daher zu optimistisch. Wir nutzen zwei einfache Werkzeuge, die das berücksichtigen:
1. **Auftrags-Bootstrap:** Für ein Intervall um eine Rate ziehen wir *Aufträge* mit Zurücklegen und berechnen die Rate neu.
2. **Permutationstest:** Um zu prüfen, ob ein Faktor (z. B. der Drucker) eine Rolle spielt, mischen wir ihn unter Aufträgen, die im anderen Faktor gleich sind (z. B. dasselbe Teil), und sehen, wie große Unterschiede allein durch Zufall entstehen.

In [ ]:
def wilson(k, n, z=1.96):
    """Wilson interval for a share k/n of independent cases (here: orders)."""
    if n == 0: return (np.nan, np.nan)
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

def order_bootstrap(df, group, cols, B=2000, level=95):
    """Rate = sum(cols) / sum(parts) per group, interval by resampling ORDERS within the group."""
    out = []
    for name, g in df.groupby(group):
        parts = g['Lot size'].to_numpy(float)
        X = g[cols].to_numpy(float)
        idx = rng.integers(0, len(g), size=(B, len(g)))
        r = X[idx].sum(axis=1) / parts[idx].sum(axis=1)[:, None]
        lo, hi = np.percentile(r, [(100 - level) / 2, 100 - (100 - level) / 2], axis=0)
        out.append(pd.DataFrame({group: name, 'defect': cols, 'rate': X.sum(axis=0) / parts.sum(), 'lo': lo, 'hi': hi, 'orders': len(g), 'parts': int(parts.sum())}))
    return pd.concat(out, ignore_index=True)

def permutation_p(stat, df, factor, strata, n=1000):
    """p-value: how often a shuffle of `factor` within `strata` gives a statistic at least as large as observed."""
    s0 = stat(df)
    sims = np.empty(n)
    for i in range(n):
        d = df.copy()
        d[factor] = d.groupby(strata)[factor].transform(lambda x: rng.permutation(x.to_numpy()))
        sims[i] = stat(d)
    return s0, (np.sum(sims >= s0) + 1) / (n + 1)

## Frage 1: Welche Teile verursachen welche Defekte?
Defektrate = fehlerhafte Teile dieser Art / alle Teile dieser Teilenummer (Teile, nicht Aufträge). Das 99-%-Intervall aus dem Auftrags-Bootstrap markiert eine Zelle als **verlässlich überdurchschnittlich**, wenn das ganze Intervall über der Gesamtrate liegt (99 %, weil wir 40 Zellen gleichzeitig betrachten).

In [ ]:
cols = DEF + ['not OK']
overall = qm[cols].sum() / qm['Lot size'].sum()
ci = order_bootstrap(qm, 'BOM Item', cols, level=99)
rate = ci.pivot(index='BOM Item', columns='defect', values='rate').loc[part_order, cols] * 100
lo = ci.pivot(index='BOM Item', columns='defect', values='lo').loc[part_order, cols] * 100
reliable = lo.gt(overall[cols] * 100, axis=1)          # whole interval above the overall rate

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(rate.values, cmap='Oranges', aspect='auto')
ax.set_xticks(range(len(cols))); ax.set_xticklabels(['breakout', 'support remains', 'under-extrusion', 'warpage', 'all scrap'], rotation=20)
ax.set_yticks(range(len(rate))); ax.set_yticklabels([f"{p} (n = {int(qm.loc[qm['BOM Item'] == p, 'Lot size'].sum())})" for p in rate.index])
for i in range(rate.shape[0]):
    for j in range(rate.shape[1]):
        ax.text(j, i, f"{rate.iat[i, j]:.1f}" + (' *' if reliable.iat[i, j] else ''), ha='center', va='center', fontsize=9, fontweight='bold' if reliable.iat[i, j] else 'normal')
ax.set_title('Defective parts per 100 parts; * = reliably above the overall rate')
plt.colorbar(im, label='% of parts'); plt.tight_layout(); plt.show()

print('overall rate (% of parts):', (overall * 100).round(2).to_dict())
print('reliably above average:', [(p, c) for p in rate.index for c in cols if reliable.loc[p, c]])

**Welche Kombinationen aus Teil und Defekt sind verlässlich, welche nur in der Tabelle sichtbar?**

**Verlässlich:** nur **Teil 1a hat mehr Warping** als der Durchschnitt (7,5 % seiner Teile gegenüber 3,8 % insgesamt; 1a wird immer als Los von 1 gedruckt, 414 Teile, diese Zelle ist daher nicht durch Cluster verzerrt). **Sichtbar, aber nicht verlässlich:** Stützstrukturreste sind am höchsten bei 8a (9,0 %), 5a (8,6 %), 6a (8,5 %) und 9a (7,5 %) gegenüber 5,3 % insgesamt; Unterextrusion bei 3a (4,6 %); Ausbruch bei 5a, 8a und 2a (2,4 bis 2,7 % gegenüber 1,3 %). Ihre Intervalle enthalten die Gesamtrate, weil wenige schlechte Aufträge mit 6 oder 12 Teilen eine Zelle dominieren. Teil 5a zeigt gar kein Warping (0,0 %), was ebenso auffällig wie unverlässlich ist.
**Grenze:** Die Daten zeigen, *welches* Teil *welchen* Defekt häufiger zeigt, nicht *warum* (Geometrie, Ausrichtung und Druckeinstellungen sind nicht erfasst). Bei 40 Zellen würden ein bis zwei „Befunde“ auf dem 99-%-Niveau allein durch Zufall auftreten; nur 1a/Warping ist ein Kandidat für eine Ursachenanalyse (zum Beispiel Ausrichtung oder BettHaftung dieses Teils).

## Frage 2: Welcher Druckfehler gehört zu welchem Drucker?
Zwei Fallen: (a) Die Drucker drucken **unterschiedliche Teilemischungen**, ein Drucker kann also nur deshalb schlecht aussehen, weil er schwierige Teile gedruckt hat; (b) die Teile eines Auftrags sind nicht unabhängig. Wir vergleichen deshalb jeden Drucker mit der bei seiner Teilemischung **erwarteten** Zahl an Defekten (beobachtet / erwartet) und testen mit einem Permutationstest, der den Drucker unter Aufträgen desselben Teils mischt.

In [ ]:
part_rate = qm.groupby('BOM Item')[DEF].sum().div(qm.groupby('BOM Item')['Lot size'].sum(), axis=0)   # defect rate per part number

def expected_by_part_mix(df):
    parts = df.groupby(['Printer', 'BOM Item'])['Lot size'].sum().unstack(fill_value=0).reindex(columns=part_rate.index, fill_value=0)
    return parts.dot(part_rate)

def printer_defect_stat(df):
    o = df.groupby('Printer')[DEF].sum()
    e = expected_by_part_mix(df)
    return (((o - e) ** 2) / e).to_numpy().sum()

obs = qm.groupby('Printer')[DEF].sum()
exp = expected_by_part_mix(qm)
print('observed defects per printer'); display(obs)
print('observed / expected given the part mix'); display((obs / exp).round(2))

ci_p = order_bootstrap(qm, 'Printer', DEF + ['not OK'], level=95)
fig, ax = plt.subplots(figsize=(9, 3.6))
for j, c in enumerate(DEF + ['not OK']):
    for k in (0, 1, 2):
        r = ci_p[(ci_p['Printer'] == k) & (ci_p['defect'] == c)].iloc[0]
        ax.errorbar(j + (k - 1) * 0.22, r['rate'] * 100, yerr=[[r['rate'] * 100 - r['lo'] * 100], [r['hi'] * 100 - r['rate'] * 100]], fmt='o', color=['#003A79', '#009EE0', '#EE7F00'][k], capsize=3, label=f'printer {k}' if j == 0 else None)
ax.set_xticks(range(5)); ax.set_xticklabels(['breakout', 'support remains', 'under-extrusion', 'warpage', 'all scrap'])
ax.set_ylabel('% of parts'); ax.set_title('Defect rate per printer with 95 % interval (order bootstrap)'); ax.legend(); plt.tight_layout(); plt.show()

s0, p = permutation_p(printer_defect_stat, qm, 'Printer', 'BOM Item')
print(f'overall test (printer vs defect profile, given the part): statistic {s0:.1f}, permutation p = {p:.2f}')

**Gibt es einen Drucker mit verlässlich anderem Defektprofil? Was sagt ein naiver Chi-Quadrat-Test, und warum führt er in die Irre?**

**Kein Drucker hat ein verlässlich anderes Defektprofil.** Beobachtet / erwartet bei gegebener Teilemischung: Drucker 0 hat mehr Warping (1,42) und Unterextrusion (1,45), aber weniger Stützstrukturreste (0,63); Drucker 1 hat wenig Unterextrusion (0,44); Drucker 2 hat wenig Warping (0,56) und mehr Stützstrukturreste (1,28). Der Permutationstest über alle Defektarten ergibt p = 0,17, und das Intervall jedes Druckers in der Grafik enthält den Durchschnitt. Die naheliegende Lesart „Drucker 0 verzieht sich, Drucker 2 nicht“ liegt im Bereich des Zufalls von Auftrag zu Auftrag.
Der **naive Chi-Quadrat-Test ergibt p = 6e-10** und würde einen starken Druckereffekt behaupten. Er führt aus zwei Gründen in die Irre: Er behandelt die 1.468 bis 1.723 Teile je Drucker als unabhängig, obwohl sie aus nur 317 bis 342 Aufträgen (Lose aus 1, 3, 6 oder 12) stammen, und er ignoriert, dass die Drucker verschiedene Teilemischungen drucken. Ein winziger p-Wert aus einem falschen Modell ist kein Beleg.
**Grenze:** Ein echter Druckereffekt mittlerer Größe könnte verborgen bleiben; die Daten sagen nur, dass sie keinen zeigen.

In [ ]:
naive = stats.chi2_contingency(qm.groupby('Printer')[DEF + ['OK']].sum().to_numpy())[1]
print(f'naive chi-square (every part independent, part mix ignored): p = {naive:.1e}')

## Frage 3: Gibt es einen Zusammenhang zwischen Extruder-Blockade und Verbindungsabbruch?
Beide Störungen sind Zeitstempel in den Fehlerdateien. Würde eine die andere auslösen oder nach sich ziehen, müssten Abbrüche **näher an Blockaden liegen, als der Zufall** sie platzieren würde. Test: Für jede Blockade den Abstand zum nächsten Verbindungsabbruch bestimmen; den Anteil innerhalb von ±24 h vergleichen mit dem Fall, dass die Abbrüche **zufällig** im selben Zeitraum liegen (1000 Ziehungen, gleiche Anzahl Abbrüche).

In [ ]:
def nearest_hours(b, c):
    """distance in hours from every time in b to the nearest time in c (both sorted datetime64 arrays)"""
    i = np.searchsorted(c, b)
    left, right = c[np.clip(i - 1, 0, len(c) - 1)], c[np.clip(i, 0, len(c) - 1)]
    return np.minimum(np.abs(b - left), np.abs(b - right)) / np.timedelta64(1, 'h')

t0, t1 = pd.Timestamp('2024-01-05'), pd.Timestamp('2025-05-14')
window = 24
rows = []
fig, axes = plt.subplots(3, 1, figsize=(10, 4), sharex=True)
for k, ax in zip((0, 1, 2), axes):
    e = errors[errors['Printer'] == k]
    blk = np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].to_numpy())
    con = np.sort(e.loc[e['Error'] == 'Connection interrupted', 'Timestamp'].to_numpy())
    ax.vlines(blk, 0.1, 0.9, color='#E2001A', label='extruder blocked'); ax.vlines(con, -0.9, -0.1, color='#009EE0', label='connection interrupted')
    ax.set_yticks([]); ax.set_ylabel(f'printer {k}')
    if k == 0: ax.legend(loc='upper right', ncol=2, fontsize=8)
    if len(blk) < 5 or len(con) < 1:
        rows.append({'printer': k, 'blockages': len(blk), 'interruptions': len(con), 'share within 24 h': np.nan, 'expected by chance': np.nan, 'p': np.nan}); continue
    share = np.mean(nearest_hours(blk, con) <= window)
    sims = []
    for _ in range(1000):
        r = np.sort(rng.integers(t0.value, t1.value, size=len(con)).astype('datetime64[ns]'))
        sims.append(np.mean(nearest_hours(blk, r) <= window))
    rows.append({'printer': k, 'blockages': len(blk), 'interruptions': len(con), 'share within 24 h': round(share, 3), 'expected by chance': round(np.mean(sims), 3), 'p': round((np.sum(np.array(sims) >= share) + 1) / 1001, 3)})
plt.tight_layout(); plt.show()
pd.DataFrame(rows).set_index('printer')

**Gibt es einen Zusammenhang? Was wäre für eine kausale Aussage nötig, und was könnte eine gemeinsame Ursache sein?**

**Kein Hinweis auf einen Zusammenhang.** Drucker 1 (34 Blockaden, 11 Abbrüche): 0 von 34 Blockaden haben einen Abbruch innerhalb von 24 h, der Zufall liefert etwa 4 % (p = 1,0, kein Überschuss). Drucker 2 (18 Blockaden, 24 Abbrüche): 3 von 18 (17 %) gegenüber 9 % durch Zufall, p = 0,20, also im Bereich des Zufalls. Drucker 0 hat nur 2 Blockaden und kann nicht getestet werden.
Das Bild zeigt eher **verschiedene Störungsmuster je Drucker**: Drucker 0 zeigt fast nur Abbrüche (29 von 35 Ereignissen), Drucker 1 fast nur Blockaden (34 von 50). Das passt zu druckerspezifischen Ursachen (Netzwerk oder Kabel bei Abbrüchen, Hotend oder Filamentweg bei Blockaden), nicht dazu, dass eine Störung die andere auslöst.
**Grenze:** Zeitliche Korrelation ist keine Kausalität, und fehlender Nachweis ist kein Nachweis des Fehlens: Mit 11 und 18 Ereignissen je Drucker wäre nur eine starke Kopplung sichtbar. Eine kausale Aussage bräuchte die Protokolle von Netzwerk und Hotend. Außerdem treten Blockaden nur beim Drucken auf, Abbrüche dagegen in jedem Zustand; das Zufallsmodell oben ignoriert das.

## Frage 4: Wie viele Teile sind gescheitert, und warum (Defektart)?
Alle geplanten Teile enden in einem von vier Zuständen. Bei gescheiterten Aufträgen ist der Grund die Störung des Druckers während des Auftrags (wie in Übung 3: Zeitfenster des Auftrags ± 5 Minuten); bei Ausschussteilen ist es die Defektart aus der QM-Datei.

In [ ]:
orders['End'] = orders['Start'] + pd.to_timedelta(orders['Actual Duration'], unit='m')
tol = pd.Timedelta(minutes=5)

def disturbance(o):
    e = errors[(errors['Printer'] == o['Printer']) & (errors['Timestamp'] >= o['Start'] - tol) & (errors['Timestamp'] <= o['End'] + tol)]
    return e['Error'].iloc[0] if len(e) else 'none'

orders['disturbance'] = orders.apply(disturbance, axis=1)
failed = orders[orders['failed'] == 1]
by_cause = failed.groupby('disturbance').agg(orders=('Order', 'size'), parts_lost=('Lot size', 'sum')).sort_values('parts_lost', ascending=False)
print('failed orders and parts lost, by disturbance during the order'); display(by_cause)
print('completed orders with a disturbance during the order:', int(((orders['failed'] == 0) & (orders['disturbance'] != 'none')).sum()))

no_qm = orders[(orders['failed'] == 0) & (~orders['Order'].isin(qm['Order']))]
fate = pd.Series({
    'OK': qm['OK'].sum(),
    'scrap: support structure remains': qm['support structure remains'].sum(),
    'scrap: warpage': qm['warpage'].sum(),
    'scrap: under-extrusion': qm['under-extrusion'].sum(),
    'scrap: material breakout': qm['material breakout'].sum(),
    'lost: failed order': failed['Lot size'].sum(),
    'unknown: completed, no QM record': no_qm['Lot size'].sum()})
planned = int(orders['Lot size'].sum())
print('planned parts:', planned, ' sum of the states:', int(fate.sum()))
fig, ax = plt.subplots(figsize=(9, 3.6))
colors = ['#7AB51D', '#EE7F00', '#EE7F00', '#EE7F00', '#EE7F00', '#E2001A', '#9AA0A6']
ax.barh(fate.index[::-1], fate.values[::-1], color=colors[::-1])
for i, v in enumerate(fate.values[::-1]): ax.text(v + 30, i, f'{int(v)}  ({v / planned * 100:.1f} %)', va='center', fontsize=9)
ax.set_xlim(0, fate.max() * 1.25); ax.set_xlabel('parts'); ax.set_title('What happens to the planned parts'); plt.tight_layout(); plt.show()

**Wie viele Teile gehen insgesamt verloren, und welche Verlustart ist größer? Was bedeuten die fehlenden QM-Einträge von 27 abgeschlossenen Aufträgen?**

Von **5.477 geplanten Teilen** sind 4.181 (76,3 %) in Ordnung. **1.296 (23,7 %) sind es nicht:**
- **572 (10,4 %) verloren in 134 gescheiterten Aufträgen**, nach Störung während des Auftrags: Verbindungsabbruch 64 Aufträge / 295 Teile, Extruder blockiert 54 / 212, Filament leer 15 / 53, Crash 1 / 12. In **keinem** abgeschlossenen Auftrag trat eine Störung auf; eine Störung während des Auftrags trennt also gescheiterte von abgeschlossenen Aufträgen.
- **604 (11,0 %) Ausschuss** in abgeschlossenen Aufträgen: Stützstrukturreste 253 (4,6 %), Warping 183 (3,3 %), Unterextrusion 104 (1,9 %), Ausbruch 64 (1,2 %).
- **120 (2,2 %) unbekannt**: 27 abgeschlossene Aufträge ohne QM-Eintrag.
Die beiden Verluste sind etwa gleich groß. Verbindungsabbrüche kosten mehr Teile (295) als Extruder-Blockaden (212), obwohl nur Blockaden ein Warnzeichen in den Energiedaten zeigen; eine Frühwarnung für Blockaden könnte daher höchstens 212 von 572 verlorenen Teilen (37 %) adressieren, noch vor Fehlalarmen. **Grenze:** Die Ursache eines gescheiterten Auftrags ist die Störung *während* des Auftrags (Zeitfenster der Fehlerdatei); warum die Störung auftrat, steht nicht in den Daten.

## Frage 5: Welche Unterschiede gibt es zwischen den Druckern?
Fünf Sichten: Auslastung und Zustände (Minutendaten), Energie beim Drucken, Störungen (Anzahl und Art), gescheiterte Aufträge, Ausschuss. Bei jedem Unterschied fragen wir: Ist er größer als der Zufall?

In [ ]:
rows, state_share = [], {}
for k in (0, 1, 2):
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', usecols=['State', 'Total_Power']).dropna(how='all')
    d['E'] = d['Total_Power'].diff() * 1000                    # Wh per minute (unit assumed kWh)
    state_share[k] = d['State'].value_counts(normalize=True).mul(100)
    rows.append({'printer': k, 'printing hours': round((d['State'] == 'printing').sum() / 60), 'energy while printing (Wh/min)': round(d.loc[d['State'] == 'printing', 'E'].mean(), 3)})
minute_view = pd.DataFrame(rows).set_index('printer')
print('states, share of time in %'); display(pd.DataFrame(state_share).round(1).loc[['printing', 'ready', 'error', 'setup', 'idle']])
display(minute_view)

types = errors.groupby(['Error', 'Printer']).size().unstack(fill_value=0)
print('disturbances (number of events)'); display(types)
print(f"chi-square type x printer (events are separate time stamps): p = {stats.chi2_contingency(types.to_numpy())[1]:.1e}")

In [ ]:
fail_rate = orders.groupby('BOM Item')['failed'].mean()

def printer_failure_stat(df):
    expected = df.groupby('Printer')['BOM Item'].apply(lambda s: fail_rate.reindex(s).sum())
    observed = df.groupby('Printer')['failed'].sum()
    return (((observed - expected) ** 2) / expected).sum()

summary = pd.DataFrame({
    'orders': orders.groupby('Printer').size(),
    'failed orders (%)': (orders.groupby('Printer')['failed'].mean() * 100).round(1),
    'scrap parts (%)': (qm.groupby('Printer')['not OK'].sum() / qm.groupby('Printer')['Lot size'].sum() * 100).round(1),
    'parts lost in failed orders (% of planned)': (orders[orders['failed'] == 1].groupby('Printer')['Lot size'].sum() / orders.groupby('Printer')['Lot size'].sum() * 100).round(1)})
display(summary)
s0, p_fail = permutation_p(printer_failure_stat, orders, 'Printer', 'BOM Item')
print(f'failed orders: printer effect given the part: permutation p = {p_fail:.2f}')

**Welche Unterschiede zwischen den Druckern sind echt, welche liegen im Bereich des Zufalls?**

**Echte Unterschiede (weit jenseits des Zufalls): das Störungsmuster.** Drucker 0 hat fast nur Verbindungsabbrüche (29 von 35 Ereignissen), Drucker 1 fast nur Extruder-Blockaden (34 von 50), Drucker 2 eine Mischung (24 Abbrüche, 18 Blockaden); Chi-Quadrat p = 6e-7. Weil Blockaden einen Drucker lange stoppen, verbringt Drucker 1 **7,2 % seiner Zeit im Zustand Fehler** gegenüber 4,0 % (Drucker 0) und 5,2 % (Drucker 2) und verliert die meiste Druckkapazität.
**Kein relevanter Unterschied:** Energie beim Drucken (1,641, 1,645, 1,643 Wh/min, bei einer Streuung von 0,14 von Minute zu Minute), Zeitanteil Drucken (32,4, 32,7, 34,3 %), Ausschussrate (13,2, 12,2, 12,5 %).
**Im Bereich des Zufalls:** gescheiterte Aufträge (9,1, 12,5, 13,0 %); bei gegebener Teilemischung ergibt der Permutationstest p = 0,20. Drucker 0 sieht am besten aus, aber 385 Aufträge können 9 % nicht von 13 % trennen.
**Grenze:** Die Drucker sind keine gleichen Experimente: Teilemischung, Zeitraum und Bediener können sich unterscheiden. „Drucker 1 ist der Problemdrucker“ stimmt für Blockaden und Zeit im Fehlerzustand, nicht für Ausschuss und nicht (verlässlich) für gescheiterte Aufträge.

## Frage 6: Welches Teil auf welchen Drucker, um Ausfälle zu minimieren?
Eine Empfehlung „Teil X auf Drucker Y drucken“ braucht einen Unterschied zwischen den Druckern **für dasselbe Teil**, der größer ist als der Zufall. Tabelle: Ausfallrate der Aufträge je Teil und Drucker mit Auftragszahl; Wilson-95-%-Intervall für den besten und den schlechtesten Drucker jedes Teils. Überlappen die Intervalle, rechtfertigen die Daten keine Präferenz.

In [ ]:
tab = orders.groupby(['BOM Item', 'Printer'])['failed'].agg(['sum', 'size']).unstack('Printer').reindex(part_order)
rows = []
for p in part_order:
    k = tab.loc[p, ('sum', slice(None))].to_numpy(); n = tab.loc[p, ('size', slice(None))].to_numpy()
    r = k / n
    best, worst = int(np.argmin(r)), int(np.argmax(r))
    lo_b, hi_b = wilson(k[best], n[best]); lo_w, hi_w = wilson(k[worst], n[worst])
    rows.append({'part': p, **{f'P{i}: failed/orders': f'{int(k[i])}/{int(n[i])}' for i in (0, 1, 2)}, 'best printer': best, 'worst printer': worst,
                 'best (%)': round(r[best] * 100, 1), 'worst (%)': round(r[worst] * 100, 1), 'intervals overlap': bool(hi_b >= lo_w)})
table6 = pd.DataFrame(rows).set_index('part')
display(table6)

by_part = orders.groupby('BOM Item')['failed'].agg(['sum', 'size']).reindex(part_order)
by_part['failed (%)'] = (by_part['sum'] / by_part['size'] * 100).round(1)
by_part['95 % interval (%)'] = [f'{lo * 100:.1f} - {hi * 100:.1f}' for lo, hi in (wilson(k, n) for k, n in zip(by_part['sum'], by_part['size']))]
print('failure rate per part, all printers together'); display(by_part.rename(columns={'sum': 'failed', 'size': 'orders'}))

fig, ax = plt.subplots(figsize=(9, 3.8))
rate6 = (tab['sum'] / tab['size'] * 100)
im = ax.imshow(rate6.values, cmap='Reds', aspect='auto', vmin=0)
ax.set_xticks(range(3)); ax.set_xticklabels([f'printer {i}' for i in range(3)]); ax.set_yticks(range(len(part_order))); ax.set_yticklabels(part_order)
for i in range(rate6.shape[0]):
    for j in range(3): ax.text(j, i, f'{rate6.iat[i, j]:.0f} %  (n = {int(tab[("size", j)].iloc[i])})', ha='center', va='center', fontsize=8, color='white' if rate6.iat[i, j] > 25 else 'black')
ax.set_title('Failed orders per part and printer'); plt.colorbar(im, label='% failed'); plt.tight_layout(); plt.show()
print(f'printer effect given the part (failed orders): permutation p = {p_fail:.2f}')

**Können Sie für ein Teil einen Drucker empfehlen? Was empfehlen Sie stattdessen, und welche Daten würden die Antwort ändern?**

**Kein Teil lässt sich begründet einem Drucker zuordnen.** Für jedes Teil überlappen die Intervalle des besten und des schlechtesten Druckers, und der Permutationstest über alle Teile ergibt p = 0,20. Die größten Abstände sind nur scheinbar: Teil 2a auf Drucker 2 (8 von 22 = 36 %) gegenüber Drucker 0 (4 von 20 = 20 %), Teil 3a auf Drucker 2 (1 von 28) gegenüber Drucker 1 (8 von 41). Bei 5 bis 40 Aufträgen je Zelle und wenigen Ausfällen entstehen solche Abstände durch Zufall.
**Was die Daten stattdessen tragen**, ist eine Aussage über die **Teile**: Ausfallrate über alle Drucker 2a 25 % (95-%-Intervall 16 bis 38 %, 59 Aufträge), 1a 16 % (13 bis 19 %, 506 Aufträge), 3a 14 %; 7a, 8a, 9a sind sehr niedrig (etwa 1 % oder weniger). Da 1a den größten Teil der Aufträge stellt, verursacht es 80 der 134 gescheiterten Aufträge. Der Hebel ist daher die **Störung** hinter den gescheiterten Aufträgen (Verbindung und Extruder, Frage 4), nicht die Zuordnung zu Druckern.
**Was die Antwort ändern würde:** mehr Aufträge je Teil und Drucker (besonders 2a, insgesamt 59 Aufträge), die Druckeinstellungen und der Wartungszustand je Auftrag und vor allem ein **geplanter Vergleich**: Dasselbe Teil abwechselnd (zufällig) zwei Druckern zuzuweisen würde die Verzerrung beseitigen, die die historische Zuordnung immer mitbringt.

## Zusammenfassung

**Zusammenfassung der sechs Antworten (Zahl, Unsicherheit, Grenze der Daten):**

| # | Antwort | Unsicherheit | Was die Daten nicht sagen |
|---|---|---|---|
| 1 | Nur **1a** zeigt mehr **Warping** (7,5 % gegenüber 3,8 %) | übrige Zellen im Bereich des Zufalls | warum (Geometrie, Ausrichtung) |
| 2 | **Kein** verlässlich anderes Defektprofil je Drucker | Permutations-p = 0,17; naiver Test irreführend | Druckereffekte mittlerer Größe |
| 3 | **Kein** Zusammenhang zwischen Blockade und Abbruch | p = 1,0 und 0,20; wenige Ereignisse | Ursachen (Netzwerk, Hotend) |
| 4 | 5.477 Teile: 76,3 % in Ordnung, 10,4 % in gescheiterten Aufträgen verloren, 11,0 % Ausschuss, 2,2 % unbekannt | exakte Zählung | warum die Störungen auftraten |
| 5 | **Störungsmuster** unterscheidet sich (Drucker 0: Verbindung, Drucker 1: Blockaden) | p = 6e-7; gescheiterte Aufträge p = 0,20 | unterschiedliche Teilemischung und Bediener |
| 6 | **Keine** Druckerzuordnung begründbar; Teile 2a und 1a sind das Problem | alle Intervalle überlappen | braucht einen randomisierten Vergleich |

**Allgemeine Lehre:** Diagnose braucht mehr als eine Tabelle. Die Teilemischung kontrollieren, beachten, dass Teile zu Aufträgen gehören, die Unsicherheit angeben und „kein Nachweis“ als Antwort akzeptieren. Die unbeantworteten *Warum*-Fragen (rote Karten) brauchen Daten, die heute nicht erfasst werden.